# Driftveil 🌌: Statistical Data Contracts for DataFrames

Welcome to the **Driftveil** interactive demo! 

Standard schema validation tools (like Pydantic, Great Expectations, or standard SQL checks) are excellent for checking data types, null values, and value ranges. However, **production machine learning models and data pipelines fail in subtler ways**: distributions shift, correlations break, and category frequencies drift. 

**Driftveil** is the statistical contract layer above your schemas. Define checks on your reference data, save them as a lightweight 2KB JSON file, and enforce them on any new batch in production.

### 1. Setup Reference and Production Data (Simulated)

Let's simulate a standard dataset with some columns that stay stable and others that drift in production.

In [ ]:
import numpy as np
import pandas as pd

# Seed for reproducibility
np.random.seed(42)
n = 2000

# 1. Reference Data (e.g., your ML training set)
ref_df = pd.DataFrame({
    "age": np.random.normal(30, 5, n),
    "income": np.random.lognormal(10, 0.5, n),
    "revenue": np.random.normal(5000, 1000, n),
    "category": np.random.choice(["A", "B", "C"], n, p=[0.5, 0.3, 0.2]),
    "clicks": np.random.randint(0, 100, n),
})
ref_df["conversions"] = ref_df["clicks"] * 0.7 + np.random.normal(0, 5, n)

# 2. Production Data (e.g., new batch from production)
new_df = pd.DataFrame({
    "age": np.random.normal(30.1, 5.05, n),                   # ✓ Stable
    "income": np.random.lognormal(10.02, 0.49, n),            # ✓ Stable
    "revenue": np.random.normal(7800, 1200, n),               # ✗ Drifted (+56% mean shift)
    "category": np.random.choice(["A", "B", "C", "D"], n, p=[0.4, 0.2, 0.2, 0.2]), # ✗ Unseen category 'D'
    "clicks": np.random.randint(0, 100, n),
})
new_df["conversions"] = np.random.normal(50, 15, n)           # ✗ Drifted (clicks/conversions relationship broken)

print("Reference and Production datasets created successfully.")

### 2. Define the Statistical Contract

Using Driftveil's fluent API, we can quickly define what distributions, correlations, and stats we expect our columns to maintain.

In [ ]:
from driftveil import DriftVeil

# Initialize with reference data
pact = DriftVeil(ref_df)

# Define column-level constraints
pact.column("age").is_normal(tolerance=0.05)
pact.column("income").is_lognormal()
pact.column("revenue").mean_stable(tolerance=0.10) # 10% maximum shift allowed
pact.column("category").no_new_categories().category_freq_stable(chi2_pvalue=0.05)

# Define pair-level constraints
pact.pair("clicks", "conversions").correlation_above(0.5)

# Define dataset-level constraints
pact.dataset.row_count_stable(tolerance=0.15)

print(f"Registered {len(pact.contracts)} contract checks on DriftVeil pact.")

### 3. Enforce and Inspect the Report

We call `.enforce()` to check the production DataFrame against the saved contract.

In [ ]:
report = pact.enforce(new_df, raise_on_fail=False)

print(report.summary())

### 4. Visualizations and Exporting Reports

Driftveil is equipped with reporting utilities. You can export interactive HTML dashboards and generate visual Matplotlib reports.

In [ ]:
# 1. Save and export the interactive HTML report (with inline base64 plots)
report.to_html("drift_report.html")
print("Saved drift_report.html. Open this file in your browser to view the interactive dashboard.")

# 2. Plot the drift overview using Matplotlib
fig = report.plot_drifts()
fig.show()

### 5. Save and Load Pact JSON

A flagship feature of Driftveil is that **you do not need the original raw reference data to run checks in production**. You can save the contract as a lightweight JSON file containing summary statistics (mean, std, quantiles, bin edges), and run checks anywhere—such as in your CI/CD pipeline or Airflow tasks.

In [ ]:
# Save pact config to disk
pact.save("production_model.pact.json")

# Load it back in a separate script or pipeline (no ref_df needed!)
loaded_pact = DriftVeil.load("production_model.pact.json")

# Enforce using the loaded configuration
loaded_report = loaded_pact.enforce(new_df)
print(f"Loaded pact successfully and executed {len(loaded_report.results)} checks.")